**I will recommmend to manually download the data from https://zenodo.org/records/18061204.** <br>
**I will make a script to remove duplicate pictures.** <br>
**Caution the data is 33 GB**

# Experiment Comparison and Training Workflow Guidelines

### 1. How to Compare Experiments Correctly
* **Direct comparisons require equal training conditions:** You cannot directly compare Model 1 trained on 10% data against Model 2 trained on 100% data to decide which architecture is better. Almost every model improves with more data.
* **Use small sizes for screening:** You can run 10 different architectures on 10% or 20% of the training data to quickly drop the bottom 5 models. Then, take only the top-performing models and re-train them on 100% of the training data to pick your ultimate winner.

### 2. Always Evaluate on the FULL Validation Set
* Even if Experiment 1 uses 10% of the training set and Experiment 2 uses 50%, both experiments must be evaluated on the exact same, full 100% validation set. 
* **Never shrink the validation set size.**

### 3. Essential Sampling Rules
* **Stratified Sampling:** When taking a 10% or 20% subset of your training data, ensure you use stratified sampling. Rock datasets are frequently imbalanced; stratified sampling guarantees that rare rock types retain their original class proportions in the smaller subset.
* **Fixed Subsets:** Keep the random seed fixed so that the 20% subset used for Model 1 contains the exact same images as the 20% subset used for Model 2.

---

## Recommended Workflow for Faster Training

| Phase | Training Data Used | Goal |
| :--- | :--- | :--- |
| **Phase 1: Fast Screening** | 10%–20% of Train | Train all 10 candidate architectures. Eliminate models that fail to converge or perform poorly. |
| **Phase 2: Hyperparameter Tuning** | 50%–100% of Train | Tune learning rates, image augmentations, and batch sizes on the promising models. |
| **Phase 3: Final Selection** | 100% of Train | Train the top 2–3 configurations on the entire training set to max out accuracy before selecting the winner on validation. |


# Why You Should Never Deploy Using Test Data



No, you should not deploy your model using the test data. The **test data** serves a single, critical purpose: to act as an **independent benchmark** that measures how well your model will perform in the real world on unseen images. Once you use it, its purpose is fulfilled. 

Here is what you should do with your data when preparing for deployment:

### 1. Keep Test Data Strictly for Evaluation
* If you include test images in your deployment pipeline or combine them into your final model's training process, you lose the ability to verify if your model actually works on new, unseen rock samples.

### 2. Retraining for Deployment (The Standard Exception)
In standard machine learning practice, there is one way you expand your dataset before deployment, but it involves the **validation set**, not the test set:

* **Step 1 (Development):** Train on Train, select best hyperparameters on Validation, and measure final true performance on Test.
* **Step 2 (Deployment Prep):** Once your model architecture and hyperparameters are 100% finalized and verified by the test score, you can optionally combine **Train + Validation** into one larger training pool. Retrain your final model on this combined set using your finalized settings.
* **Step 3 (Deployment):** Deploy this newly retrained model to serve real users or real-world inference.

### 3. Test Data is Your "Firewall"
* Think of the test set as your **safety net**. 
* In deployment, your model will encounter real-world challenges—different lighting, novel camera angles, or unexpected backgrounds. 
* The test score gives you your only realistic expectation of how the model will handle these real-world conditions.


# 0. Get Data

Because the data set is too big, its better to download it yourself. at https://zenodo.org/records/18061204. Unzip it and put it the data folder


The three folders contain carbonate-rock images captured using three different microscope illumination modes:

* **`PPL-1223` — Plane-Polarized Light:** Uses one polarizing filter. It emphasizes natural color, texture, grain boundaries, pores, and alteration.
* **`XPL-1223` — Cross-Polarized Light:** Uses two polarizers oriented perpendicular to each other. It reveals interference colors, crystal orientation, twinning, and other mineral optical properties.
* **`R-1223` — Reflected Light:** Light is reflected from the specimen’s surface rather than transmitted through it. This is useful for opaque minerals and surface features.

> **⚠️ Important Modeling Note**  
> These represent three distinct *imaging modalities* of the dataset—not three different rock classifications. A model trained on one mode should generally not silently mix in another mode unless you are deliberately designing a multimodal training pipeline.

You are going to get something like this below. We are only going to work on the PPL_1223 folder

```text

📁 carbonate_1223
├── 📁 PPL-1223
│   ├── 🔴 train
│   ├── 🟡 val
│   └── 🟢 test
│
├── 📁 XPL-1223
│   ├── 🔴 train
│   ├── 🟡 val
│   └── 🟢 test
│
└── 📁 R-1223
    ├── 🔴 train
    ├── 🟡 val
    └── 🟢 test
```

# 1. Delete Duplicate

#### A lot of the images are duplicated cross the train/val/test folder and across the classes folders. This code will remove it

In [8]:
from pathlib import Path
from collections import defaultdict
import hashlib
import pandas as pd

DATASET_ROOT = Path("data/carbonate_1223/PPL-1223")

# Safety switch: change to False only after reviewing the output.
DRY_RUN = True

# When a duplicate occurs across splits, keep the copy in the
# highest-priority split. This preserves test/validation independence
# by removing matching copies from training.
SPLIT_PRIORITY = {
    "test": 0,
    "val": 1,
    "train": 2,
}


def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """Calculate a file's SHA-256 hash without loading it all into memory."""
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        while chunk := file.read(chunk_size):
            sha256.update(chunk)

    return sha256.hexdigest()


def get_split(file_path):
    """Return train, val, or test from a dataset path."""
    for part in file_path.parts:
        if part in SPLIT_PRIORITY:
            return part

    raise ValueError(f"Could not determine split for {file_path}")


def get_class(file_path):
    """Return the class-folder name."""
    return file_path.parent.name


valid_extensions = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}

image_paths = sorted(
    path
    for path in DATASET_ROOT.rglob("*")
    if path.is_file() and path.suffix.lower() in valid_extensions
)

print(f"Hashing {len(image_paths):,} images...")

hash_groups = defaultdict(list)

for number, image_path in enumerate(image_paths, start=1):
    file_hash = calculate_sha256(image_path)
    hash_groups[file_hash].append(image_path)

    if number % 1_000 == 0:
        print(f"Processed {number:,}/{len(image_paths):,}")


duplicate_groups = {
    file_hash: paths
    for file_hash, paths in hash_groups.items()
    if len(paths) > 1
}

files_to_delete = []
report_rows = []

for file_hash, paths in duplicate_groups.items():

    ordered_paths = sorted(
        paths,
        key=lambda path: (
            SPLIT_PRIORITY[get_split(path)],
            str(path),
        )
    )

    keeper = ordered_paths[0]
    duplicates = ordered_paths[1:]

    classes = {get_class(path) for path in paths}
    splits = {get_split(path) for path in paths}

    for duplicate in duplicates:
        files_to_delete.append(duplicate)

        report_rows.append({
            "sha256": file_hash,
            "keep": str(keeper),
            "delete": str(duplicate),
            "keep_split": get_split(keeper),
            "delete_split": get_split(duplicate),
            "keep_class": get_class(keeper),
            "delete_class": get_class(duplicate),
            "class_conflict": len(classes) > 1,
            "cross_split": len(splits) > 1,
        })

duplicate_report = pd.DataFrame(report_rows)

print(f"Duplicate groups: {len(duplicate_groups):,}")
print(f"Files marked for deletion: {len(files_to_delete):,}")
print(
    "Files with conflicting class assignments:",
    f"{duplicate_report['class_conflict'].sum():,}"
)
print(
    "Duplicates crossing dataset splits:",
    f"{duplicate_report['cross_split'].sum():,}"
)

# Display only a small sample in Jupyter
display(duplicate_report.head(20))

# Save the complete report without printing it
duplicate_report.to_csv("duplicate_report.csv", index=False)
print("Full report saved to duplicate_report.csv")

if DRY_RUN:
    print("Dry run enabled—nothing was deleted.")
else:
    deleted_count = 0

    for file_path in files_to_delete:
        if file_path.exists():
            file_path.unlink()
            deleted_count += 1

    print(f"Deleted {deleted_count:,} duplicate files.")

Hashing 25,688 images...
Processed 1,000/25,688
Processed 2,000/25,688
Processed 3,000/25,688
Processed 4,000/25,688
Processed 5,000/25,688
Processed 6,000/25,688
Processed 7,000/25,688
Processed 8,000/25,688
Processed 9,000/25,688
Processed 10,000/25,688
Processed 11,000/25,688
Processed 12,000/25,688
Processed 13,000/25,688
Processed 14,000/25,688
Processed 15,000/25,688
Processed 16,000/25,688
Processed 17,000/25,688
Processed 18,000/25,688
Processed 19,000/25,688
Processed 20,000/25,688
Processed 21,000/25,688
Processed 22,000/25,688
Processed 23,000/25,688
Processed 24,000/25,688
Processed 25,000/25,688
Duplicate groups: 0
Files marked for deletion: 0


KeyError: 'class_conflict'